## Identity and Position 

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer

ALPHABET = "ء أ ؤ إ ئ ا ب ت ة ث ج ح خ د ذ ر ز س ش ص ض ط ظ ع غ ف ق ك ل م ن ه و ى ي"
MAX_LENGTH = 10


def indexes(my_list, desired_element):
    return [i + 1 for i, element in enumerate(my_list)
            if element == desired_element]


def input_id_pos(input_word, max_length=MAX_LENGTH):
    if len(input_word) > max_length:
        raise ValueError(
            f"'{input_word}' has {len(input_word)} letters; "
            f"maximum is {max_length}."
        )

    # 1) Identity: 35 dimensions
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts([ALPHABET])

    sequences = tokenizer.texts_to_sequences(list(input_word))
    flat = [value for sequence in sequences for value in sequence]

    identity = [
        '1' if index in flat else '0'
        for index in range(1, len(tokenizer.word_index) + 1)
    ]

    # 2) Position: 12 bits per letter, up to 12 letters
    
    unique_letters = sorted(set(input_word))
    positions = []

    for character in unique_letters:
        for position in indexes(list(input_word), character):
            value = 1 << (max_length - position)
            bits = f'{value:0{max_length}b}'
            positions.extend(bits)

    target_size = max_length * max_length
    positions.extend(['0'] * (target_size - len(positions)))

    return identity, positions


def save_encodings(word_list, filename, max_length=MAX_LENGTH):
    id_rows = []
    pos_rows = []

    for word in word_list:
        identity, position = input_id_pos(word, max_length)
        id_rows.append(identity)
        pos_rows.append(position)

    df_word = pd.DataFrame({'word': word_list})
    df_id = pd.DataFrame(id_rows)
    df_pos = pd.DataFrame(pos_rows)

    df_id.columns = range(1, df_id.shape[1] + 1)       # 1–35
    df_pos.columns = range(1, df_pos.shape[1] + 1)    

    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df_word.to_excel(writer, sheet_name='word', index=True)
        df_id.to_excel(writer, sheet_name='identity', index=True)
        df_pos.to_excel(writer, sheet_name='position', index=True)

In [ ]:
words= pd.read_excel("PONG/filtered_words/words.xlsx")['word'].to_list()
tl_words= pd.read_excel("PONG/filtered_words/TL_words.xlsx")['TL_word'].to_list()
dl_words= pd.read_excel("PONG/filtered_words/DL_words.xlsx")['DL_word'].to_list()

In [ ]:
save_encodings(words,  'PONG/filtered_words/TL_words_idpos.xlsx')
save_encodings(tl_words,  'PONG/filtered_words/TL_words_idpos.xlsx')
save_encodings(dl_words,  'PONG/filtered_words/dl_words_idpos.xlsx')

## Combined encodings

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical


MAX_LENGTH = 10

letters = "ءأؤإئابتةثجحخدذرزسشصضطظعغفقكلمنهوىي"

#  one-hot lookup
y_train_label = list(range(len(letters)))
y_train_label.reverse()

y_train_label_onehot = to_categorical(
    y_train_label,
    num_classes=len(letters)
)


def indexes(my_list, desired_element):
    return [
        i + 1
        for i, element in enumerate(my_list)
        if element == desired_element
    ]


def input_id_pos(input_word):
    #  Identity encoding: 35 dimensions
    tokenizer = Tokenizer()

    fit_text = (
        "ء أ ؤ إ ئ ا ب ت ة ث ج ح خ د ذ ر ز س ش ص ض "
        "ط ظ ع غ ف ق ك ل م ن ه و ى ي"
    )

    tokenizer.fit_on_texts([fit_text])

    sequences = tokenizer.texts_to_sequences(input_word)
    flat = [
        value
        for sequence in sequences
        for value in sequence
    ]

    identity = [
        1 if index in flat else 0
        for index in range(1, len(tokenizer.word_index) + 1)
    ]

    # Position encoding: 14 × 14 = 196 dimensions
    unique_letters = sorted(set(input_word))

    position_flat = []

    for character in unique_letters:
        for position in indexes(list(input_word), character):
            position_flat.append(position)

    position = []

    for letter_position in position_flat:
        value = 1 << (MAX_LENGTH - letter_position)
        bits = f"{value:0{MAX_LENGTH}b}"
        position.extend(int(bit) for bit in bits)

    target_length = MAX_LENGTH * MAX_LENGTH

    if len(position) < target_length:
        position.extend(
            [0] * (target_length - len(position))
        )

    return identity, position


def onehotencod(word):
    # 35 × 14 = 490 dimensions
    encoded = []

    for character in word:
        position = letters.find(character)

        # Ignore characters not in the original alphabet
        if position < 0:
            continue

        encoded.append(
            y_train_label_onehot[position].tolist()
        )

    flat = [
        int(value)
        for onehot_block in encoded
        for value in onehot_block
    ]

    target_length = len(letters) * MAX_LENGTH

    if len(flat) < target_length:
        flat.extend([0] * (target_length - len(flat)))

    return flat


def save_encodings(word_list, output_file):
    identity_rows = []
    position_rows = []
    onehot_rows = []

    for word in word_list:
        identity, position = input_id_pos(word)

        identity_rows.append(identity)
        position_rows.append(position)
        onehot_rows.append(onehotencod(word))

    # words
    df_words = pd.DataFrame({
        "word": word_list
    })

    # identity encodings
    df_identity = pd.DataFrame(identity_rows)
    df_identity.columns = range(
        1, df_identity.shape[1] + 1
    )

    #  position encodings
    df_positions = pd.DataFrame(position_rows)
    df_positions.columns = range(
        1, df_positions.shape[1] + 1
    )

    # one-hot encodings
    df_onehot = pd.DataFrame(onehot_rows)
    df_onehot.columns = [
        f"v{i}" for i in range(df_onehot.shape[1])
    ]

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        df_words.to_excel(
            writer,
            sheet_name="words",
            index=True
        )

        df_identity.to_excel(
            writer,
            sheet_name="identity",
            index=True
        )

        df_positions.to_excel(
            writer,
            sheet_name="position",
            index=True
        )

        df_onehot.to_excel(
            writer,
            sheet_name="onehot",
            index=True
        )

    print(f"Saved to {output_file}")

In [3]:
words= pd.read_excel("PONG/filtered_words/data/words.xlsx")['word'].to_list()
tl_words= pd.read_excel("PONG/filtered_words/data/TL_words.xlsx")['TL_word'].to_list()
dl_words= pd.read_excel("PONG/filtered_words/data/DL_words.xlsx")['DL_word'].to_list()
unr_words= pd.read_excel("PONG/filtered_words/data/unr_words.xlsx")['unr_word'].to_list()


In [4]:
save_encodings(words, 'PONG/filtered_words/data/words_encodings.xlsx')
save_encodings(tl_words, 'PONG/filtered_words/data/TL_words_encodings.xlsx')
save_encodings(dl_words, 'PONG/filtered_words/data/DL_words_encodings.xlsx')
save_encodings(unr_words, 'PONG/filtered_words/data/unr_words_encodings.xlsx')


Saved to PONG/filtered_words/data/words_encodings.xlsx
Saved to PONG/filtered_words/data/TL_words_encodings.xlsx
Saved to PONG/filtered_words/data/DL_words_encodings.xlsx
Saved to PONG/filtered_words/data/unr_words_encodings.xlsx
